# Final Project: Multi-Agent Research Assistant

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/project.html

**How to use:** run the cells from the top (Runtime → Run all, or Shift+Enter cell by cell). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need a key: add it with the 🔑 Secrets panel on the left, or set `os.environ["OPENAI_API_KEY"] = "..."` in a cell. Examples that use Ollama need a local Ollama server and will not work in Colab. For GPU examples choose Runtime → Change runtime type → T4 GPU.

In [ ]:
# Libraries this lesson uses that are not preinstalled in Colab
%pip install -q langgraph

## 6. Shared State Schema

In [ ]:
from typing import List, Optional
from typing_extensions import TypedDict


class ResearchState(TypedDict):
    # Input
    topic: str
    sub_questions: List[str]

    # Research Agent
    raw_sources: List[dict]            # {url, title, content, score}

    # Classification Agent
    classified_sources: List[dict]     # {url, title, category, relevance}

    # NER Agent
    entities: List[dict]               # {text, label, source_url, frequency}
    entity_relationships: List[dict]   # {entity1, entity2, relation, context}

    # Browser Agent (LAM)
    browser_results: List[dict]        # {url, action, extracted_data}

    # Analyzer Agent
    organized_findings: List[dict]     # {theme, findings, sources, entities}
    outline: str

    # Writer Agent
    draft: str

    # Illustration Agent
    illustrations: List[dict]          # {section, prompt, image_path}

    # Critic Agent
    critic_feedback: Optional[str]
    critic_score: Optional[int]
    revision_count: int
    final_report: Optional[str]

    # Course-topic artefacts
    transformer_config: dict
    moe_analysis: Optional[str]
    status: str

## Test code for the course environment

**Minimal LangGraph skeleton with the Critic revision loop**

In [ ]:
# 1. Minimal LangGraph skeleton with the Critic revision loop
#    pip install langgraph typing_extensions
from typing import List, Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class ResearchState(TypedDict, total=False):
    topic: str
    draft: str
    critic_score: Optional[int]
    critic_feedback: Optional[str]
    revision_count: int
    final_report: Optional[str]

def writer(state):
    n = state.get("revision_count", 0)
    return {"draft": f"Draft v{n+1} on '{state['topic']}'"}

def critic(state):
    score = 5 + state.get("revision_count", 0)          # toy critic: improves each round
    return {"critic_score": score, "critic_feedback": "add citations" if score < 7 else "ok",
            "revision_count": state.get("revision_count", 0) + 1}

def route(state):
    if state["critic_score"] >= 7 or state["revision_count"] >= 3:
        return "finalize"
    return "writer"

def finalize(state):
    return {"final_report": state["draft"]}

g = StateGraph(ResearchState)
g.add_node("writer", writer); g.add_node("critic", critic); g.add_node("finalize", finalize)
g.add_edge(START, "writer"); g.add_edge("writer", "critic")
g.add_conditional_edges("critic", route, {"writer": "writer", "finalize": "finalize"})
g.add_edge("finalize", END)
app = g.compile()

print(app.invoke({"topic": "Mixture of Experts", "revision_count": 0}))

**NER Agent prototype with spaCy  (pip install spacy && python -m spacy download en_core_web**

In [ ]:
# 2. NER Agent prototype with spaCy  (pip install spacy && python -m spacy download en_core_web_sm)
import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")
text = ("In 2017 Vaswani and colleagues at Google introduced the Transformer. "
        "OpenAI later released GPT-4, while Mistral AI built Mixtral, a mixture-of-experts model.")
doc = nlp(text)

entities = [{"text": e.text, "label": e.label_} for e in doc.ents]
print(entities)
print(Counter(e["label"] for e in entities))

**Classification Agent prototype using a zero-shot transformer**

In [ ]:
# 3. Classification Agent prototype using a zero-shot transformer
from transformers import pipeline

clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
categories = ["technical", "review", "tutorial", "opinion", "dataset"]
source = "This step-by-step guide shows how to fine-tune DistilBERT for emotion classification."
out = clf(source, candidate_labels=categories)
print(dict(zip(out["labels"], [round(s, 3) for s in out["scores"]])))

**Competition score: average of 3 topics over the 100-point rubric**

In [ ]:
# 4. Competition score: average of 3 topics over the 100-point rubric
rubric = {"report_quality": 20, "source_coverage": 15, "entity_extraction": 10,
          "source_classification": 10, "illustrations": 10, "speed": 10,
          "cost_efficiency": 5, "error_handling": 5, "innovation": 15}

def score_report(earned_fraction: dict) -> float:
    return sum(rubric[k] * earned_fraction.get(k, 0.0) for k in rubric)

topic_scores = [score_report({k: f for k in rubric}) for f in (0.75, 0.8, 0.9)]
print("per topic:", topic_scores, "average:", round(sum(topic_scores) / 3, 1))